In [21]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [22]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds2'

JSON_EXTENSION = "fintoc4.json"

In [23]:
if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())[:5]

In [24]:
import json

import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree
from pager.doc_model.dtype import Document, Page
from pager import Region

from typing import Dict

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [36]:
from pager.page_model.sub_models.extractors import RegionSorterCutXYExtractor
def custom_sort(rows):
    if len(rows) <= 1:
        return rows
    sorter = RegionSorterCutXYExtractor()
    bboxes = [row.segment for row in rows]
    for i, bbox in enumerate(bboxes):
        bbox.id = i
    order = sorter.sort_using_XYCut(bboxes)
    return [rows[i] for i in order]


class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, regions, output_model:LogicTreeModel):
        reg_page = [[]]
        tmp_page = 0
        for reg in regions:
            if tmp_page == reg['metainfo']["page"]:
                reg_page[tmp_page].append(reg)
            else:
                reg_page.append([reg])
                tmp_page += 1
            
        document = Document([Page(regions=[Region(r) for r in regs], num_page=num) for num, regs in enumerate(reg_page)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree

class ExtractTree:
    def __init__(self):
        self.creater = CustomJson2LogicTree()
        self._font_from_doc = {}
        self._list_font_type = []

        
    def extract(self, tree:LogicTreeModel):
        doc_rows = [row for page in tree.document.pages for row in custom_sort(page.rows)]
        row_num_page = [page.num_page for page in tree.document.pages for _ in page.rows]
        clusters = self.get_cluster(doc_rows)
        level_header_cluster = self.get_level_header(clusters)
        level_header_cluster = self.gpt_filter(level_header_cluster, clusters)
        regions = self.get_regions(doc_rows, row_num_page, level_header_cluster)
        self.creater.upload(regions, tree)

    def gpt_filter(self, level_header_cluster, clusters):
        K = 5
        """{cl:level+1}"""
        print("""
Ниже приведен набор строк подозрительных на заголовок с их предположительным уровнем (отсортированных)
необходимо сказать, какие из них действительно могут являться заголовками и выдать их предположительный уровень. 
Ответ выдать в виде списка, где будут перечисленны номер кластера (только заголовки), отсортированные по уровню.

Ответ представь в виде списка Python List[int] формате, без лишнего текста. Номера кластеров 
Пример ответа: "[1, 23, 3, 2]"

Данные:
""")
        for index, level in level_header_cluster.items():
            print(f'Кластер {index} (уровень {level})')
            cls = clusters[index]
            count = min(K, len(cls))
            for i in range(count):
                print('текст:', cls[i].text)

        res = input("Ответ: ")
        ans = {ind: i+1 for i, ind in enumerate(eval(res))}
    
        return ans
        
    def get_regions(self, doc_rows, row_num_page, level_header_cluster):
        num_cluster = [ self._font_from_doc[vec] for vec in self._list_font_type]
        regions = []
        
        def get_region(row, cl, num_page):
            tmp_reg = {
                "rows": [row],
                "label": "header" if cl in level_header_cluster else "text",
                "metainfo": {"page": num_page}
            }
            if cl in level_header_cluster:
                tmp_reg["header_level"] = level_header_cluster[cl]
            return tmp_reg
        
        tmp_cl = num_cluster[0]
        tmp_num_page = 0
        tmp_reg = get_region(doc_rows[0].to_dict(), tmp_cl, tmp_num_page)
        old_row = doc_rows[0]
        for row, cl, num_page in zip(doc_rows[1:], num_cluster[1:], row_num_page[1:]):
            dist = row.segment.get_min_dist(old_row.segment)
            h = (row.segment.height+old_row.segment.height)*0.75  #Полуторный интервал
            if cl == tmp_cl and num_page==tmp_num_page and dist<h:
                tmp_reg['rows'].append(row.to_dict())
            else:
                regions.append(tmp_reg.copy())
                tmp_cl = cl
                tmp_num_page = num_page
                tmp_reg = get_region(row.to_dict(), cl,num_page)
        return regions
        
    def get_level_header(self, clusters):
        main_clust, main_font = self.get_main_cluster(clusters)
        header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > main_font}
        header_cluster_index = list(header_cluster.keys())
        header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)        
        level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
        return level_header_cluster
        
    def get_main_cluster(self, clusters):
        max_clust = -1
        max_size = 0
        for ind, rows in clusters.items():
            if max_size < len(rows):
                max_clust = ind
                max_size = len(rows)
        font_text =  clusters[max_clust][0].font

        font_text.to_dict()
        return max_clust, font_text

    def get_cluster(self, doc_rows):
        self._font_from_doc = {}
        self._list_font_type = []
        def get_vec_feature(row):
            fd = row.font.to_dict()
            name = fd['name']
            size = int(round(fd['size']/2))
            width = int(round(fd['width']/0.1))
            italic = int(round(fd['italic']/0.1))
            return (name, size, width, italic)
        
        def get_id_exist_cluster(feature):
            self._list_font_type.append(feature)
            if feature in self._font_from_doc.keys():
                 return self._font_from_doc[feature] 
            return None
                
        def add_new_cluster(feature):
            indexs= list(self._font_from_doc.values())
            index = max(indexs)+1 if len(indexs) != 0 else 0
            self._font_from_doc[feature] = index
            clusters[index] = []
            return index
            
        clusters = {}
        for row in doc_rows:
            feature_row = get_vec_feature(row)
            index = get_id_exist_cluster(feature_row)
            if index is None:
                index = add_new_cluster(feature_row)
            clusters[index].append(row)
        return clusters

extract_tree=ExtractTree() 

In [37]:
def tree2fintoc_json(doc_tree) -> Dict:
    fintoc_json = []
    for id_node, node in doc_tree['nodes']['regions'].items():
        if node['label'] == 'header':
            reg = {
                "text":node['text'],
                "id": id_node,
                "page": node['metainfo']['page'] + 1, # В PageR специально оставил нумерацию с нуля (поскольку номер определяется не порядком, а пишится на странице и это еще не реализовано)
                "depth": node['header_level']
            }
            fintoc_json.append(reg)
    return fintoc_json

N = len(FILE_PATHS)
for i, file_path in enumerate(FILE_PATHS):    
    pdf_reader.read_from_file(file_path)
    pdf_reader.extract()
    pdf2tree.convert(pdf_reader, tree)
    extract_tree.extract(tree)
    
    doc_tree = tree.to_dict()
    pred_path = FINTOC_EN_PRED/(file_path.name+'.'+JSON_EXTENSION)
    # print(tree.document.md)
    
    rez = tree2fintoc_json(doc_tree)
    with open(pred_path, 'w') as f:
        json.dump(rez, f)

    print(f'{i+1}/{N} ({(i+1)/N*100:.2f}%)')

/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



Ниже приведен набор строк подозрительных на заголовок с их предположительным уровнем (отсортированных)
необходимо сказать, какие из них действительно могут являться заголовками и выдать их предположительный уровень. 
Ответ выдать в виде списка, где будут перечисленны номер кластера (только заголовки), отсортированные по уровню.

Ответ представь в виде списка Python List[int] формате, без лишнего текста. Номера кластеров 
Пример ответа: "[1, 23, 3, 2]"

Данные:

Кластер 2 (уровень 1)
текст: VALARTIS RUSSIAN MARKET FUND
Кластер 5 (уровень 2)
текст: PROSPECTUS
Кластер 7 (уровень 3)
текст: CONTENTS
текст: 1.
текст: MANAGEMENT AND ADMINISTRATION
текст: 2.
текст: DEFINITIONS
Кластер 6 (уровень 4)
текст: 25 SEPTEMBER 2012
текст: 5.1.
текст: Typical investors’ Profile
текст: 5.2.
текст: Risk factors and Investment Restrictions
Кластер 10 (уровень 5)
текст: Board of Directors
текст: Conducting
текст: Officers
текст: (Délégués à la
текст: Investment
Кластер 11 (уровень 6)
текст: gestion journal

Ответ:  [2, 5, 7, 6, 10]


1/5 (20.00%)

Ниже приведен набор строк подозрительных на заголовок с их предположительным уровнем (отсортированных)
необходимо сказать, какие из них действительно могут являться заголовками и выдать их предположительный уровень. 
Ответ выдать в виде списка, где будут перечисленны номер кластера (только заголовки), отсортированные по уровню.

Ответ представь в виде списка Python List[int] формате, без лишнего текста. Номера кластеров 
Пример ответа: "[1, 23, 3, 2]"

Данные:

Кластер 23 (уровень 1)
текст: RAM (LUX) SYSTEMATIC FUNDS –
текст: EUROPEAN EQUITIES
текст: RAM (LUX) SYSTEMATIC FUNDS – NORTH
текст: AMERICAN EQUITIES
текст: RAM (LUX) SYSTEMATIC FUNDS –
Кластер 0 (уровень 2)
текст: RAM (LUX) SYSTEMATIC FUNDS
текст: RAM (LUX) SYSTEMATIC FUNDS
текст: Articles of Association
Кластер 1 (уровень 3)
текст: Luxembourg SICAV with multiple sub-funds
текст: RAM (LUX) SYSTEMATIC FUNDS
текст: Sub-fund factsheets
Кластер 2 (уровень 4)
текст: PROSPECTUS
текст: &
текст: ARTICLES OF ASSOCIATION
К

Ответ:  [23, 0, 1, 12, 28, 7, 8]


...

4/5 (80.00%)


Cannot set gray non-stroke color because /'P0' is an invalid float value



Ниже приведен набор строк подозрительных на заголовок с их предположительным уровнем (отсортированных)
необходимо сказать, какие из них действительно могут являться заголовками и выдать их предположительный уровень. 
Ответ выдать в виде списка, где будут перечисленны номер кластера (только заголовки), отсортированные по уровню.

Ответ представь в виде списка Python List[int] формате, без лишнего текста. Номера кластеров 
Пример ответа: "[1, 23, 3, 2]"

Данные:

Кластер 2 (уровень 1)
текст: Sales prospectus
текст: Additional information for investors
текст: in Germany
Кластер 1 (уровень 2)
текст: Investment company under Luxembourg law
текст: (the “Company”)
текст: November 2011
текст: Applicability of the provisions of this sales prospectus
текст: of 2010“).
Кластер 0 (уровень 3)
текст: UBS (Lux) Equity SICAV
Кластер 3 (уровень 4)
текст: Shares in the Company may be acquired on the basis of this sales prospectus, the
текст: Only the information contained in the sales prospectus and in

Ответ:  [2, 0, 20, 9]


5/5 (100.00%)


In [38]:
import subprocess
import pandas as pd
subprocess.run(['python', 'metric.py', '--gt_folder', FINTOC_EN_ANNOTS, '--submission_folder', FINTOC_EN_PRED])
df = pd.read_csv('td_report.csv', delimiter='\t')
df[-2:-1][['Prec', 'Rec', 'F1']]

,Prec,Rec,F1
21,0.40494009494440963,0.4655079829895737,0.3649085905774653


In [39]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-4:-3]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
25,25.0,18.8,17.2,34.9,40.8


In [ ]:
25.6	25.4	19.6	43.5	39.8